# Demographic Data Pipeline

---
## Overview
Notebook 1 of 8.

This notebook demonstrates a Bronze layer ingestion pattern for demographic data.

Run the entire notebook to load the data.

---

## Source Data

- **File:** `demog.csv`
- **Location:** A Public GitHub repository set up especially for this exercise
- **Note:** Uses HTTPS URL so the notebook runs across any Databricks workspace

---

## Processing Steps

**1. Environment Setup**  
Ensures the required catalog and schemas exist for the Bronze layer

**2. Data Ingestion**  
Loads source data into a DataFrame  
*Note: Uses Pandas rather than Spark to reliably read from the GitHub HTTPS URL*

**3. Provenance Metadata**  
Enriches the data with lineage tracking:
- Ingestion timestamp
- Unique ingest ID  
- Source filename

---

## Target

Data is written to a **Delta table in the Bronze layer**, using a Spark MERGE operation for full upsert capability.

---

In [0]:
from pyspark.sql.functions import lit, current_timestamp
from delta.tables import DeltaTable
from datetime import datetime
import pandas as pd

In [0]:
#global variables
#do not change the github link - this is a public repo from where the source data can be reliably sourced into any databricks workspace
CATALOG = "cluepoints_db_demo"
SCHEMA = "bronze"
GITHUB_URL = "https://raw.githubusercontent.com/prolfdog/cp_db_demo/main/"
ingestion_id = datetime.now().strftime('%Y%m%d_%H%M%S')
loaded_at = current_timestamp()
table_name = "demog"
full_table_name = f"{CATALOG}.{SCHEMA}.{table_name}"

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

DataFrame[]

In [0]:
#use pandas to import from github as spark can't read https urls - NOT FOR PRODUCTION, ONLY FOR DEMO
#in production use spark.read.csv
#drop unnamed column on import
df_demog = spark.createDataFrame(
    pd.read_csv(f"{GITHUB_URL}demog.csv").drop(columns=['Unnamed: 0'])
)

In [0]:
#add data provenance variables to the data frame
df_demog = (df_demog 
    .withColumn("source_file", lit("demog.csv"))
    .withColumn("loaded_at", lit(loaded_at))
    .withColumn("ingestion_id", lit(ingestion_id))
)

In [0]:
if not spark.catalog.tableExists(full_table_name):
    #create the table if it exists
    df_demog.write.format("delta").saveAsTable(full_table_name)
else:
    #from analysing the data we know that subjid is what makes a row unique
    #so we upsert data on subjid column
    #the target is the existing delta table, the source is the dataframe
    #note this merge will overwrite blindly even if nothing has changed - 
    #if we would wish to preserve the original ingestion_id for unchanged data then we would merge on a row checksum instead
    target = DeltaTable.forName(spark, full_table_name)
    target.alias("target").merge(df_demog.alias("source"),"target.subjid = source.subjid").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
#display the table
display(spark.sql(f"select * from {full_table_name} limit 10"))

subjid,age_n,birthdate,birthdatetime,birthtime,consda,race,sex,center,country,rand,source_file,loaded_at,ingestion_id
528183037,61,1942-05-12,1942-05-12 00:18:51,00:18:51,2004-06-30,1.0,1.0,183,USA,0,demog.csv,2026-01-05T08:24:42.854Z,20260105_082437
528183058,71,1933-07-24,1933-07-24 16:23:08,16:23:08,2005-04-11,1.0,1.0,183,USA,1,demog.csv,2026-01-05T08:24:42.854Z,20260105_082437
528183023,71,1933-08-31,1933-08-31 15:50:57,15:50:57,2004-03-12,1.0,2.0,183,USA,0,demog.csv,2026-01-05T08:24:42.854Z,20260105_082437
528183060,73,1932-04-02,1932-04-02 15:03:12,15:03:12,2005-04-18,1.0,1.0,183,USA,0,demog.csv,2026-01-05T08:24:42.854Z,20260105_082437
528183039,60,1944-06-18,1944-06-18 00:28:32,00:28:32,2004-09-20,1.0,1.0,183,USA,0,demog.csv,2026-01-05T08:24:42.854Z,20260105_082437
528183062,68,1936-11-13,1936-11-13 13:37:41,13:37:41,2005-04-25,1.0,2.0,183,USA,0,demog.csv,2026-01-05T08:24:42.854Z,20260105_082437
528183029,71,1932-10-11,1932-10-11 10:17:56,10:17:56,2004-05-18,1.0,1.0,183,USA,0,demog.csv,2026-01-05T08:24:42.854Z,20260105_082437
528183064,64,1941-02-28,1941-02-28 19:58:36,19:58:36,2005-06-08,1.0,1.0,183,USA,1,demog.csv,2026-01-05T08:24:42.854Z,20260105_082437
528183019,78,1925-06-09,1925-06-09 08:39:20,08:39:20,2004-02-16,1.0,1.0,183,USA,1,demog.csv,2026-01-05T08:24:42.854Z,20260105_082437
528183020,77,1926-03-24,1926-03-24 06:48:41,06:48:41,2004-02-20,1.0,1.0,183,USA,0,demog.csv,2026-01-05T08:24:42.854Z,20260105_082437
